In [ ]:
import h5py
import pickle
import numpy as np
import matplotlib.pyplot as plt
from cell_data import calculate_purity
import behave_ana
from scipy.stats import zscore,entropy

In [ ]:
hdf_file = h5py.File('/home/zyyk78/HDD/zyyk78/Stage3/C01/MS__2025-03-24T14_59_12_20250516_153537_fr_dec_corr_20250516_193223_results_refine.hdf5', 'r') #来自Caiman的结果
pkl_file = ''  #对应的行为数据包
data = np.array(hdf_file['estimates']['C'])
data_01 = zscore(data, axis=1)
hdf_file.close()
with open(pkl_file,"rb") as pkl_f:
        Event=pickle.load(pkl_f)

In [ ]:
# sort_with_MS=sorted(Event,ignore_list)
keep_list=['MinS','MinT','SlnL','SlnR','ROmL','ROmR']

ignore_list=[i for i in list(Event.keys()) if i not in keep_list]
long_event=['MinT','Blackhole','MinS']
eve_list=behave_ana.sort_events(Event,ignore_list=ignore_list,long_event=long_event)
for i in eve_list:
    if eve_list[0][1] != 'MinT_S':
        eve_list.pop(0)
    else:
        eve_list.pop(0)
        break
for i in eve_list:
    if eve_list[-1][1] != 'MinT_E':
        eve_list.pop(-1)
    else:
        eve_list.pop(-1)
        break

In [ ]:
trigger_Rew=[]
trigger_Rom=[]
trigger_TL=[]
trigger_TR=[]
trigger_All=[]

frame_count=0
for _,eve in eve_list:
    if eve=='MinS_S' or eve=='MinS_E':
        frame_count+=1
        continue
    else:
        trigger_All.append(frame_count)
          
    if eve=='SlnL' or eve=='SlnR':
        trigger_Rew.append(frame_count)
    if eve=='ROmL' or eve =='ROmR':
        trigger_Rom.append(frame_count)   
    if eve=='SlnL' or eve == 'ROmL':
        trigger_TL.append(frame_count)
    if eve=='SlnR' or eve == 'ROmR':
        trigger_TR.append(frame_count)
    if not eve in keep_list:
        print('Error ',eve)
trigger_list_E=   [trigger_Rew,trigger_Rom,trigger_TL,trigger_TR,trigger_All]        
print(frame_count)
print(data_01.shape[1])

matrix_list_E=[]
matrix_pur_list_E=[]
matrix_E_list_E=[]
for trigger in trigger_list_E:
    react_list=[]
    for i in trigger:
        crop_data=np.array(data_01[:,max(i//4 - 12,0):min(i//4 + 13,data_01.shape[1])])   
        if crop_data.shape[1] == 25:
            react_list.append(crop_data)

    react_list_stack=np.stack(react_list)
    mean_responses = np.mean(react_list_stack, axis=0)
    En,purity = calculate_purity(react_list_stack)
    matrix_list_E.append(mean_responses)
    matrix_pur_list_E.append(purity)
    matrix_E_list_E.append(En)
    
#################################################    
trigger_Sw_S=[]
trigger_Sw_E=[]
frame_count=0
past_side=None
past_count=None
for _,eve in eve_list:
    
    if eve=='MinS_S' or eve=='MinS_E':
        frame_count+=1
        continue

    curr_side = 'L' if eve=='SlnL' or eve=='ROmL' else 'R'
    if curr_side != past_side and past_side!=None and past_count!=None:
        trigger_Sw_E.append(frame_count)
        trigger_Sw_S.append(past_count)
    past_side = curr_side
    past_count = frame_count
    
trigger_list_S=   [trigger_Sw_S,trigger_Sw_E]        

matrix_list_S=[]
matrix_pur_list_S=[]
for trigger in trigger_list_S:
    react_list=[]
    for i in trigger:
        crop_data=np.array(data_01[:,max(i//4 - 31,0):min(i//4 + 32,data_01.shape[1])])
        if crop_data.shape[1] == 63:
            react_list.append(crop_data)
        else:
            print('skip',crop_data.shape[1])

    react_list_stack=np.stack(react_list)
    mean_responses = np.mean(react_list_stack, axis=0)
    _,purity = calculate_purity(react_list_stack)
    matrix_list_S.append(mean_responses)
    matrix_pur_list_S.append(purity)
#################################################
matrix_list_E=np.array(matrix_list_E)
matrix_pur_list_E=np.array(matrix_pur_list_E)
matrix_E_list_E=np.array(matrix_E_list_E)

matrix_list_S=np.array(matrix_list_S)
matrix_pur_list_S=np.array(matrix_pur_list_S)

trigger_list=trigger_list_E+trigger_list_S

In [ ]:
mix_data=[]
for (mat,pur) in zip(matrix_list_E,matrix_pur_list_E):
    mix_data.append(mat*pur)
    
for (mat,pur) in zip(matrix_list_S,matrix_pur_list_S):
    mix_data.append(mat*pur)
    
tot_mix_data = np.concatenate(mix_data, axis=1)
features, sorted_idx,cluster,Z=cluster_matrix_hierarchical(tot_mix_data,max_features=20,n_clusters=2,metric='correlation',linkage_method='average')

In [ ]:
import matplotlib.gridspec as gridspec
from scipy.stats import sem  # 标准误差
titles = ["Reward", "Unreward", "Trial L", "Trial R", "Trial All","Switsh_S", "Switsh_E"]
lens=[2,2,2,2,2,5,5]
width_r=[1,1,1,1,1,2.5,2.5]

plt.rcParams.update({
    'font.size': 16,         # 所有字体的默认大小
    'axes.titlesize': 18,    # 子图标题大小
    'axes.labelsize': 16,    # 坐标轴标签字体
    'xtick.labelsize': 18,   # x 轴刻度字体
    'ytick.labelsize': 14,   # y 轴刻度字体
    'legend.fontsize': 14,   # 图例字体
    'figure.titlesize': 20   # 整体大标题字体
})



# ========== 构建 Figure ==========
fig = plt.figure(figsize=(25, 14), dpi=300)
gs = gridspec.GridSpec(2, 8, height_ratios=[2, 1], width_ratios=[1]+width_r, wspace=0.05, hspace=0.05)

# ========== 上方：dendrogram 和 heatmap ==========
last_merge_distances = sorted(Z[:, 2], reverse=True)  # 所有合并距离（降序）
cut_height = last_merge_distances[features[0] - 2]     # 直接取第n_clusters-1大的距离

ax_dendro = plt.subplot(gs[0, 0])
axes_heat = [plt.subplot(gs[0, i+1]) for i in range(7)]

dendro_data = dendrogram(
    Z,
    ax=ax_dendro,
    orientation='left',
    color_threshold=cut_height,
    no_labels=True
)
ax_dendro.axvline(x=cut_height, color='r', linestyle='--', linewidth=1.5,)
ax_dendro.set_xticks([])
ax_dendro.set_yticks([])
ax_dendro.set_title('Dendrogram')
# ========== 分配标签 ==========
labels = fcluster(Z, features[0], criterion='maxclust')  # 每个神经元所属簇
sorted_idx = dendro_data['leaves'][::-1]  # 已排序的神经元索引

from collections import defaultdict

cluster_colors = defaultdict(lambda: 'gray')
leaves = dendro_data['leaves']        # 序号排序
leaves_colors = dendro_data['leaves_color_list']  # 每个leaf的颜色

# 构建 cluster_id -> color 的映射
for idx, leaf in enumerate(leaves):
    cluster_id = cluster[leaf]
    if cluster_id not in cluster_colors:
        cluster_colors[cluster_id] = leaves_colors[idx]

for i, (matrix, ax) in enumerate(zip(mix_data, axes_heat)):
    im = ax.imshow(matrix[sorted_idx], cmap='bwr', vmin=-0.7, vmax=0.7, aspect='auto')
    ax.axvline(x=matrix.shape[1]//2, color='k', linestyle='--', linewidth=1)
    ax.set_yticks([])
    ax.set_title(titles[i]+f'\nn_event={len(trigger_list[i])}')
    ax.set_xticks([])

# ========== colorbar ==========


# ========== 下方：每个簇的平均响应 ± SEM ==========
axes_avg = [plt.subplot(gs[1, i+1]) for i in range(7)]

for i, (matrix, ax) in enumerate(zip(mix_data, axes_avg)):
    x = np.linspace(-lens[i], lens[i], matrix.shape[1])
    for cluster_id in range(1, features[0]+1):
        cluster_idx = np.where(labels == cluster_id)[0]
        mean_trace = matrix[cluster_idx].mean(axis=0)
        error = sem(matrix[cluster_idx], axis=0)

        ax.plot(x, mean_trace, label=f'Cluster {cluster_id}', color=cluster_colors[cluster_id])
        ax.fill_between(x, mean_trace-error, mean_trace+error, color=cluster_colors[cluster_id], alpha=0.3)
    
    ax.axvline(x=0, color='k', linestyle='--', linewidth=1)
    ax.set_yticks([])
    ax.set_xlabel(f'Time: ±{lens[i]} s')
    ax.set_xticks([(x[0]+x[-1])/2])
    ax.set_xticklabels([0])
    ax.set_xlim(x[0], x[-1])
    ax.set_ylim(-0.5, 0.5)
    ax.axhline(y=0, color='red', linewidth=1)
axes_avg[0].set_yticks([-0.5,0,0.5])
plt.tight_layout(rect=[0, 0, 1, 1], pad=0.5)
cbar_ax = fig.add_axes([0.35, -0.02, 0.3, 0.02])  # [left, bottom, width, height]
cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal', label='Mean_Resp * Purity')
plt.savefig('rsc1.pdf',format='pdf')